# Austin, TX parcels

Build an Austin city-only parcel export by joining two free Travis Central
Appraisal District (TCAD) sources — the same tabular-values + parcel-geometry
join pattern used for Houston.

**Source 1 — appraisal values (`PROP.TXT`, fixed-width).** TCAD's free appraisal
roll export (`traviscad.org/publicinformation` → `wp-content/largefiles/...zip`) is a
True Automation / PACS 'Legacy 8.0.32' export. The master property file `PROP.TXT`
is **fixed-width** with one record per property+owner. Field byte-offsets below are
from `TP_Legacy8.0.32-AppraisalExportLayout` (Property sheet). It carries the
homestead/non-homestead land + improvement value split, `market_value`,
`land_state_cd` (Texas state class), `py_owner_name`, `situs_city`, and an
`ex_exempt` total-exemption flag. **⚠️ traviscad.org is Cloudflare-gated** — download
the ZIP in a browser and place it at `DATA_DIR/tcad-appraisal-export.zip`.

**Source 2 — parcel geometry (taxmaps ArcGIS).**
`https://taxmaps.traviscountytx.gov/arcgis/rest/services/Parcels/MapServer/0`
(373k+ parcels, paginated, EPSG:4326). Provides `PROP_ID`, `hyperlink`, and polygon
geometry. (Its value fields are null — that's why we join `PROP.TXT`.)

**Join:** appraisal values ← geometry on `prop_id`.

**v1 scope:** TCAD = Travis County; filter to the Austin city boundary via osmnx.
Ships Travis-County parcels only (Austin's Williamson/Hays slivers are a follow-up).

**PMTiles:** ~250k+ Austin parcels — use PMTiles for performance.

In [ ]:
import io
import os
import sys
import glob
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from shapely.ops import unary_union

sys.path.append('..')
from parcel_calculations import add_improvement_ratio_fields, classify_property_refined
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DOWNLOAD_DATA = 0   # set to 1 to (re)download geometry from ArcGIS
DATA_DIR = 'data/austin'
os.makedirs(DATA_DIR, exist_ok=True)

# --- Source 1: TCAD appraisal roll export (browser-download, Cloudflare-gated) --
APPRAISAL_ZIP_PATH = os.path.join(DATA_DIR, 'tcad-appraisal-export.zip')
APPRAISAL_EXTRACT_DIR = os.path.join(DATA_DIR, 'appraisal_export')

# PROP.TXT fixed-width fields we need — (name, start1based, end1based) from the
# TP_Legacy8.0.32 Property layout. Converted to 0-based half-open colspecs below.
PROP_FIELDS = [
    ('prop_id',            1,   12),
    ('prop_type_cd',       13,  17),
    ('sup_num',            23,  34),
    ('geo_id',             547, 596),
    ('py_owner_name',      609, 678),
    ('situs_city',         1110, 1139),
    ('land_hstd_val',      1796, 1810),
    ('land_non_hstd_val',  1811, 1825),
    ('imprv_hstd_val',     1826, 1840),
    ('imprv_non_hstd_val', 1841, 1855),
    ('appraised_val',      1916, 1930),
    ('assessed_val',       1946, 1960),
    ('ex_exempt',          2671, 2671),
    ('legal_acreage',      1660, 1675),
    ('land_acres',         2772, 2791),
    ('land_state_cd',      2742, 2751),
    ('market_value',       4214, 4227),
]
PROP_COLSPECS = [(s - 1, e) for (_, s, e) in PROP_FIELDS]
PROP_NAMES    = [n for (n, _, _) in PROP_FIELDS]

# --- Source 2: TCAD parcel geometry (ArcGIS taxmaps) ------------------------
#   query URL = {GEOM_BASE}/{GEOM_DATASET}/{GEOM_SERVICE}/{GEOM_LAYER}/query
GEOM_BASE    = 'https://taxmaps.traviscountytx.gov/arcgis/rest/services'
GEOM_DATASET = 'Parcels'
GEOM_SERVICE = 'MapServer'
GEOM_LAYER   = 0
GEOM_CACHE_PATH = os.path.join(DATA_DIR, 'austin-tx-geometry.parquet')

## 1a. Extract the appraisal export and locate PROP.TXT

Download the export ZIP from `traviscad.org/publicinformation` in a browser and save
it as `DATA_DIR/tcad-appraisal-export.zip` (Cloudflare blocks scripted downloads).
The export contains many big fixed-width files; we only need `PROP.TXT` (the master
property record). Extracting `PROP.TXT` alone keeps disk use down.

In [ ]:
if not os.path.exists(APPRAISAL_ZIP_PATH):
    raise FileNotFoundError(
        f'Appraisal ZIP not found at {APPRAISAL_ZIP_PATH}.\n'
        'Download the appraisal roll export from https://traviscad.org/publicinformation '
        '(e.g. "2026 Preliminary Appraisal Roll Export") in a browser and save it there.'
    )

os.makedirs(APPRAISAL_EXTRACT_DIR, exist_ok=True)
prop_txt_path = os.path.join(APPRAISAL_EXTRACT_DIR, 'PROP.TXT')

with zipfile.ZipFile(APPRAISAL_ZIP_PATH) as zf:
    names = zf.namelist()
    prop_name = next((n for n in names if os.path.basename(n).upper() == 'PROP.TXT'), None)
    if prop_name is None:
        raise FileNotFoundError(f'PROP.TXT not found in ZIP. Entries: {names}')
    if not os.path.exists(prop_txt_path):
        print(f'Extracting {prop_name} ...')
        with zf.open(prop_name) as src, open(prop_txt_path, 'wb') as dst:
            while True:
                chunk = src.read(1 << 24)
                if not chunk:
                    break
                dst.write(chunk)
    else:
        print('PROP.TXT already extracted.')

print(f'PROP.TXT: {os.path.getsize(prop_txt_path):,} bytes')

## 1b. Parse PROP.TXT (fixed-width)

Read only the columns we need, in chunks (the file is multi-GB). Keep certified
records (`sup_num == 0`) for real property (`prop_type_cd == 'R'`), dedupe to one
row per `prop_id`, and derive canonical value fields:
`land_value = land_hstd + land_non_hstd`, `improvement_value = imprv_hstd + imprv_non_hstd`.

In [ ]:
CHUNK = 200_000
VALUE_COLS = ['land_hstd_val', 'land_non_hstd_val', 'imprv_hstd_val',
              'imprv_non_hstd_val', 'appraised_val', 'assessed_val', 'market_value']

kept = []
total = 0
reader = pd.read_fwf(
    prop_txt_path, colspecs=PROP_COLSPECS, names=PROP_NAMES,
    dtype=str, encoding='latin-1', chunksize=CHUNK,
)
for ch in reader:
    total += len(ch)
    for c in PROP_NAMES:
        ch[c] = ch[c].astype(str).str.strip()
    ch['sup_num'] = pd.to_numeric(ch['sup_num'], errors='coerce')
    ch = ch[(ch['sup_num'] == 0) & (ch['prop_type_cd'].str.upper() == 'R')]
    if len(ch):
        kept.append(ch)
print(f'Scanned {total:,} PROP.TXT records')
prop_df = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame(columns=PROP_NAMES)
print(f'Real + certified records: {len(prop_df):,}')

for c in VALUE_COLS:
    prop_df[c] = pd.to_numeric(prop_df[c], errors='coerce').fillna(0)

prop_df['prop_id'] = pd.to_numeric(prop_df['prop_id'], errors='coerce').astype('Int64')
prop_df = prop_df.dropna(subset=['prop_id']).drop_duplicates('prop_id')

prop_df['_land_value'] = prop_df['land_hstd_val'] + prop_df['land_non_hstd_val']
prop_df['_imp_value']  = prop_df['imprv_hstd_val'] + prop_df['imprv_non_hstd_val']
prop_df['_market']     = prop_df['market_value'].where(
    prop_df['market_value'] > 0, prop_df['_land_value'] + prop_df['_imp_value'])

# Assessor REPORTED land size (4 implied decimals); prefer land_acres, else legal_acreage.
# Used as the $/sqft denominator (robust to fragment/sliver GIS polygons); GIS area is the
# fallback when reported is missing. See the canonical-fields cell.
prop_df['land_acres_ac']  = pd.to_numeric(prop_df['land_acres'],   errors='coerce') / 10000.0
prop_df['legal_acres_ac'] = pd.to_numeric(prop_df['legal_acreage'], errors='coerce') / 10000.0
prop_df['reported_ac']    = prop_df['land_acres_ac'].where(prop_df['land_acres_ac'] > 0, prop_df['legal_acres_ac'])

appraisal = prop_df[[
    'prop_id', '_land_value', '_imp_value', '_market', 'reported_ac',
    'land_state_cd', 'py_owner_name', 'situs_city', 'ex_exempt',
]].copy()
print(f'Unique properties: {len(appraisal):,}')
print('Land value > 0:', int((appraisal['_land_value'] > 0).sum()))
print('Improvement value > 0:', int((appraisal['_imp_value'] > 0).sum()))
pd.set_option('display.max_columns', None)
display(appraisal.head(5))

## 1c. Pull parcel geometry from the taxmaps ArcGIS layer

Paginated pull in EPSG:4326 (~373k parcels; a few minutes). Cached to parquet.
`get_feature_data_with_geometry` keeps each polygon's outer ring (holes/multipart
simplified) — fine for parcel footprints.

In [ ]:
if DOWNLOAD_DATA == 1 or not os.path.exists(GEOM_CACHE_PATH):
    print('Downloading TCAD parcel geometry (paginated)...')
    geom_gdf = get_feature_data_with_geometry(
        dataset_name=GEOM_DATASET, base_url=GEOM_BASE, layer_id=GEOM_LAYER,
        paginate=True, out_epsg=4326, service_type=GEOM_SERVICE, verbose=True,
    )
    if geom_gdf is None or len(geom_gdf) == 0:
        raise RuntimeError('Geometry pull returned no features — check the ArcGIS endpoint.')
    geom_gdf.to_parquet(GEOM_CACHE_PATH, index=False)
    print(f'Cached geometry -> {GEOM_CACHE_PATH}')
else:
    print(f'Using cached geometry: {GEOM_CACHE_PATH}')
    geom_gdf = gpd.read_parquet(GEOM_CACHE_PATH)

print(f'Geometry rows: {len(geom_gdf):,} | CRS={geom_gdf.crs}')
print('Geometry columns:', geom_gdf.columns.tolist())
display(geom_gdf.drop(columns='geometry').head(3))

## 2. Join appraisal values onto parcel geometry (on prop_id)

Both sources key on the TCAD property id (`PROP_ID` in the geometry layer,
`prop_id` in PROP.TXT). Normalize both to integers and merge.

In [ ]:
g_id   = next((c for c in ['PROP_ID', 'prop_id', 'property_id'] if c in geom_gdf.columns), None)
g_link = next((c for c in ['hyperlink', 'link', 'url'] if c in geom_gdf.columns), None)
if g_id is None:
    raise KeyError(f'No PROP_ID in geometry layer. Columns: {geom_gdf.columns.tolist()}')
print(f'geometry id column: {g_id} | link column: {g_link}')

geom_gdf = geom_gdf.copy()
geom_gdf['prop_id'] = pd.to_numeric(geom_gdf[g_id], errors='coerce').astype('Int64')
geom_gdf = geom_gdf.dropna(subset=['prop_id'])
if g_link and g_link != 'hyperlink':
    geom_gdf = geom_gdf.rename(columns={g_link: 'hyperlink'})

parcel_gdf = geom_gdf.merge(appraisal, on='prop_id', how='left')
matched = parcel_gdf['_land_value'].notna().sum()
print(f'Joined {len(parcel_gdf):,} parcels | with appraisal match: {matched:,} '
      f'({matched / max(len(parcel_gdf),1) * 100:.1f}%)')

# Canonical working fields used downstream (Houston naming).
parcel_gdf['land_val']     = parcel_gdf['_land_value']
parcel_gdf['bld_val']      = parcel_gdf['_imp_value']
parcel_gdf['tot_appr_val'] = parcel_gdf['_market']
parcel_gdf['state_class']  = parcel_gdf['land_state_cd']
parcel_gdf['mailto']       = parcel_gdf['py_owner_name']
parcel_gdf['city']         = parcel_gdf['situs_city']
parcel_gdf['acct']         = parcel_gdf['prop_id'].astype(str)
print('Columns:', [c for c in parcel_gdf.columns if not c.startswith('_')])

## 3. Inspect state_class distribution

In [ ]:
with pd.option_context('display.max_rows', 80):
    print(parcel_gdf['state_class'].value_counts(dropna=False).head(80))

## 4. Restrict to the Austin city boundary

TCAD covers Travis County. Filter to the **authoritative City of Austin full-purpose
jurisdiction** boundary (`data.austintexas.gov` BOUNDARIES_jurisdictions, dissolved
from its 35 polygons) via a centroid-within test — preferred over osmnx geocoding,
which only approximates the limits. osmnx remains a fallback if the service is down.

In [ ]:
import io, requests
from shapely.ops import unary_union

if parcel_gdf.crs is None:
    parcel_gdf = parcel_gdf.set_crs('EPSG:4326')
elif parcel_gdf.crs.to_epsg() != 4326:
    parcel_gdf = parcel_gdf.to_crs('EPSG:4326')

# Authoritative city limits = City of Austin FULL PURPOSE jurisdiction polygons
# (data.austintexas.gov BOUNDARIES_jurisdictions). Preferred over osmnx geocoding,
# which returns an approximate boundary that lags annexations.
JURIS_URL = ('https://services.arcgis.com/0L95CJ0VTaxqcmED/ArcGIS/rest/services/'
             'BOUNDARIES_jurisdictions/FeatureServer/0/query')
params = {'where': "JURISDICTION_TYPE='FULL'", 'outFields': 'JURISDICTION_TYPE',
          'outSR': 4326, 'f': 'geojson'}
print('Fetching Austin FULL PURPOSE jurisdiction boundary (authoritative city limits)...')
try:
    resp = requests.get(JURIS_URL, params=params, timeout=120)
    resp.raise_for_status()
    juris = gpd.read_file(io.BytesIO(resp.content))
    juris = juris.set_crs('EPSG:4326') if juris.crs is None else juris.to_crs('EPSG:4326')
    boundary_geom = unary_union(list(juris.geometry))
    print(f'Loaded {len(juris)} full-purpose polygons -> dissolved {boundary_geom.geom_type}')
except Exception as e:
    print(f'Jurisdiction fetch failed ({e}); falling back to osmnx geocode.')
    import osmnx as ox
    boundary_geom = ox.geocode_to_gdf('Austin, Texas, USA').to_crs('EPSG:4326').geometry.iloc[0]

# Authoritative spatial filter only. NOTE: situs_city is unreliable (blank for much
# of downtown/commercial), so a city-name pre-filter wrongly drops in-city parcels —
# verified it halved the result and stripped most commercial. Rely on the boundary.
# Centroids are computed in a projected CRS (3857) for correctness.
print('Applying spatial filter to Austin boundary...')
n_before = len(parcel_gdf)
valid = parcel_gdf['geometry'].notnull() & parcel_gdf['geometry'].apply(lambda g: getattr(g, 'is_valid', False))
cent = parcel_gdf.loc[valid, 'geometry'].to_crs(3857).centroid.to_crs(4326)
inside = valid.copy()
inside[valid] = cent.within(boundary_geom)
parcel_gdf = parcel_gdf[inside].copy()
print(f'Before: {n_before:,} | After: {len(parcel_gdf):,} | Removed: {n_before - len(parcel_gdf):,}')
print(f'Bounds: {parcel_gdf.total_bounds}')

In [ ]:
# Sanity check: Austin bounds ~ minx -98.0, maxx -97.4, miny 30.0, maxy 30.7
bounds = parcel_gdf.total_bounds
assert -98.2 < bounds[0] < -97.5, f'Unexpected minx: {bounds[0]}'
assert -97.9 < bounds[2] < -97.3, f'Unexpected maxx: {bounds[2]}'
assert  29.9 < bounds[1] < 30.4,  f'Unexpected miny: {bounds[1]}'
assert  30.2 < bounds[3] < 30.8,  f'Unexpected maxy: {bounds[3]}'
print('✅ Bounds sanity check passed')

## 5. Handle duplicate parcel ids (condos, multi-record parcels)

In [ ]:
acct_col = 'acct'
n_dupes = parcel_gdf.duplicated(subset=[acct_col], keep=False).sum()
print(f'Duplicate rows by {acct_col}: {n_dupes:,}')
if n_dupes > 0:
    print('Collapsing duplicates: summing value fields, unioning geometries...')
    sum_cols = [c for c in ['tot_appr_val', 'land_val', 'bld_val'] if c in parcel_gdf.columns]
    cat_cols = [c for c in parcel_gdf.columns if c not in set(sum_cols + ['geometry', acct_col])]
    agg = {c: 'sum' for c in sum_cols}; agg.update({c: 'first' for c in cat_cols})
    collapsed = parcel_gdf.groupby(acct_col, dropna=False).agg(agg).reset_index()
    geom_union = parcel_gdf.groupby(acct_col, dropna=False)['geometry'].apply(
        lambda gs: unary_union([g for g in gs if g is not None]) if any(g is not None for g in gs) else None
    )
    collapsed['geometry'] = geom_union.values
    parcel_gdf = gpd.GeoDataFrame(collapsed, geometry='geometry', crs=parcel_gdf.crs)
    print(f'✅ Rows after collapse: {len(parcel_gdf):,}')
else:
    print('✅ No duplicates')

## 6. Classify property type using Texas state codes

Texas `state_class` codes are statewide, so the Houston mapping applies to Austin.

In [ ]:
def categorize_property_type(state_class_val):
    raw = str(state_class_val or '').strip().upper()
    if raw in ('X',):                              return 'Exempt'
    if raw in ('W',):                              return 'Government / State'
    if raw in ('A1', 'A2'):                        return 'Single Family'
    if raw in ('B1', 'B2', 'B3', 'B4'):            return 'Multifamily'
    if raw in ('C1', 'C2', 'C3'):                  return 'Vacant Residential'
    if raw in ('D1', 'D2', 'E1', 'E2', 'E3'):      return 'Agricultural / Rural'
    if raw == 'F1':                                return 'Commercial'
    if raw == 'F2':                                return 'Industrial'
    if raw.startswith('G'):                        return 'Mineral / Oil & Gas'
    if raw.startswith('J') or raw.startswith('U'): return 'Utility'
    if raw in ('L1', 'L2', 'M1', 'O1', 'S'):       return 'Personal Property / Inventory'
    return 'Other'

parcel_gdf['PROPERTY_CATEGORY'] = parcel_gdf['state_class'].apply(categorize_property_type)
with pd.option_context('display.max_rows', None):
    print(parcel_gdf['PROPERTY_CATEGORY'].value_counts(dropna=False))

## 7. Exclude exempt parcels

Drop parcels flagged totally exempt in PROP.TXT (`ex_exempt == 'T'`), state-class
exempt/government (`X`/`W`), or matching an Austin-area public-owner keyword.
Homestead exemptions do NOT fully exempt a parcel.

In [ ]:
export_gdf = parcel_gdf.copy()
exempt_by_state = export_gdf['PROPERTY_CATEGORY'].isin(['Exempt', 'Government / State'])
exempt_by_flag  = export_gdf.get('ex_exempt', pd.Series('', index=export_gdf.index)).astype(str).str.upper().eq('T')
print(f'Exempt by PROP.TXT ex_exempt flag: {int(exempt_by_flag.sum()):,}')

GOVT_KEYWORDS = [
    'CITY OF AUSTIN', 'TRAVIS COUNTY', 'STATE OF TEXAS', 'AUSTIN ISD', 'AISD',
    'AUSTIN COMMUNITY COLLEGE', 'AUSTIN COMM COLL', 'CENTRAL HEALTH',
    'LOWER COLORADO RIVER', 'LCRA', 'AUSTIN WATER', 'CAPITAL METRO', 'CAP METRO',
    'UNIVERSITY OF TEXAS', 'UNIV OF TEXAS', 'UNITED STATES', 'US GOVT', 'U.S. GOVERNMENT',
]
pattern = '|'.join(GOVT_KEYWORDS)
exempt_by_owner = export_gdf['mailto'].astype(str).str.upper().str.contains(pattern, na=False)
print(f'Exempt by ownership keyword: {int(exempt_by_owner.sum()):,}')

export_gdf['exemption_flag'] = (exempt_by_state | exempt_by_flag | exempt_by_owner).astype(int)
print(export_gdf['exemption_flag'].value_counts())
before = len(export_gdf)
export_gdf = export_gdf[export_gdf['exemption_flag'] == 0].copy()
print(f'Removed {before - len(export_gdf):,} exempt parcels | remaining: {len(export_gdf):,}')

## 8. Refined land use classification

In [ ]:
export_gdf['property_land_use_category'] = export_gdf['PROPERTY_CATEGORY']
non_geo = {'Mineral / Oil & Gas', 'Personal Property / Inventory'}
before = len(export_gdf)
export_gdf = export_gdf[~export_gdf['property_land_use_category'].isin(non_geo)].copy()
print(f'Dropped {before - len(export_gdf):,} non-geographic records')

export_gdf['land_value']        = pd.to_numeric(export_gdf.get('land_val', np.nan), errors='coerce')
export_gdf['improvement_value'] = pd.to_numeric(export_gdf.get('bld_val',  np.nan), errors='coerce')

# Refined classification (Vacant / Underdeveloped / Parking Lot); sf_cutoff=0.67 is the
# Texas default from Houston. Fetches Overture footprints for no-improvement candidates.
export_gdf['property_land_use_refined'] = classify_property_refined(export_gdf)
print(export_gdf['property_land_use_refined'].value_counts(dropna=False))

## 9. Compute canonical fields

In [ ]:
from pyproj import Geod
geod = Geod(ellps='WGS84')

def geodesic_area_sqft(geom):
    if geom is None or geom.is_empty:
        return np.nan
    if geom.geom_type == 'Polygon':
        lon, lat = geom.exterior.coords.xy
        area_m2, _ = geod.polygon_area_perimeter(lon, lat)
        return abs(area_m2) * 10.763910416709722
    if geom.geom_type == 'MultiPolygon':
        return sum(geodesic_area_sqft(p) for p in geom.geoms)
    return np.nan

SQFT_PER_ACRE = 43560.0
export_gdf['geometry'] = export_gdf['geometry'].apply(lambda g: g if g is None or g.is_valid else g.buffer(0))
print('Computing parcel areas...')
export_gdf['gis_area_sqft'] = export_gdf['geometry'].apply(geodesic_area_sqft)
export_gdf.loc[export_gdf['gis_area_sqft'] < 1, 'gis_area_sqft'] = np.nan

# Denominator for all per-sqft metrics: assessor REPORTED land size when present (robust to
# undersized/fragment GIS polygons on large tracts), else the GIS polygon area. Verified GIS
# approx reported for ~99% of parcels, so this only moves genuine artifacts.
export_gdf['reported_sqft'] = pd.to_numeric(export_gdf.get('reported_ac', np.nan), errors='coerce') * SQFT_PER_ACRE
_use_reported = export_gdf['reported_sqft'] > 0
export_gdf['land_area_sqft'] = np.where(_use_reported, export_gdf['reported_sqft'], export_gdf['gis_area_sqft'])
export_gdf['area_source']    = np.where(_use_reported, 'reported', 'gis')
export_gdf['land_area_acres'] = export_gdf['land_area_sqft'] / SQFT_PER_ACRE
# Flag tiny fractional remnants/slivers: < 500 sqft can't be a standalone lot, so its
# $/sqft (assessor value / tiny area) is meaningless and renders as a false spike.
export_gdf['likely_remnant'] = (export_gdf['land_area_sqft'] < 500).astype(int)

export_gdf['full_market_value'] = pd.to_numeric(export_gdf.get('tot_appr_val', np.nan), errors='coerce')
export_gdf['land_value']        = pd.to_numeric(export_gdf.get('land_val', np.nan), errors='coerce')
export_gdf['improvement_value'] = pd.to_numeric(export_gdf.get('bld_val',  np.nan), errors='coerce')
_den = export_gdf['land_area_sqft'].replace(0, np.nan)
export_gdf['full_market_value_per_sqft'] = export_gdf['full_market_value'] / _den
export_gdf['land_value_per_sqft']        = export_gdf['land_value']        / _den
export_gdf['improvement_value_per_sqft'] = export_gdf['improvement_value'] / _den

export_gdf = add_improvement_ratio_fields(export_gdf, land_col='land_value', improvement_col='improvement_value')
print('Canonical fields computed')

## 10. Build parcel detail link

In [ ]:
# Prefer the geometry layer's hyperlink (TCAD/Prodigy property-detail URL);
# else build one from the parcel id.
if 'hyperlink' in export_gdf.columns and export_gdf['hyperlink'].notna().any():
    export_gdf['link'] = export_gdf['hyperlink'].astype(str).str.replace(
        'stage.travis.prodigycad.com', 'travis.prodigycad.com', regex=False)
else:
    export_gdf['link'] = 'https://travis.prodigycad.com/property-detail/' + export_gdf['acct'].astype(str).str.strip()
print(export_gdf['link'].head())

## 11. Select and export canonical parquet

In [ ]:
COLUMNS_TO_EXPORT = [
    'geometry', 'exemption_flag',
    'property_land_use_category', 'property_land_use_refined',
    'full_market_value', 'full_market_value_per_sqft',
    'land_value', 'land_value_per_sqft',
    'improvement_value', 'improvement_value_per_sqft',
    'TLLDIMPROV', 'IMPR_LAND_RATIO', 'IMPR_LAND_PCT', 'IMPR_PCT_TOTAL',
    'link',
    'land_area_acres',   # QC: land area used as the $/sqft denominator
    'area_source',       # QC: 'reported' (assessor land_acres) or 'gis' polygon fallback
    'likely_remnant',    # QC: 1 = tiny (<500 sqft) fractional remnant, muted in the viz
]
for col in COLUMNS_TO_EXPORT:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[COLUMNS_TO_EXPORT].rename(columns={'land_value': 'current_full_land_value'})
export_final['geometry'] = export_final['geometry'].apply(lambda g: g if g is None or g.is_valid else g.buffer(0))
export_final = gpd.GeoDataFrame(export_final, geometry='geometry', crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs('EPSG:4326')

canonical_path = os.path.join(DATA_DIR, 'austin-tx-parcels.parquet')
today_str = datetime.now().strftime('%Y_%m_%d')
dated_path = os.path.join(DATA_DIR, f'austin-tx-parcels_{today_str}.parquet')
export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)
print(f'✅ Saved canonical parquet: {canonical_path}')
print(f'Total rows exported: {len(export_final):,}')
print(export_final['property_land_use_refined'].value_counts(dropna=False))
print('Bounds:', export_final.total_bounds)

## 11b. Validate the exported parquet (13 checks; exit 0 = pass)

In [ ]:
import subprocess
from pathlib import Path
project_root = Path.cwd()
while project_root.parent != project_root and not (project_root / 'data' / 'scripts' / 'validate_city_parquet.py').exists():
    project_root = project_root.parent
validator = project_root / 'data' / 'scripts' / 'validate_city_parquet.py'
abs_parquet = str((Path.cwd() / canonical_path).resolve())
res = subprocess.run([sys.executable, str(validator), abs_parquet], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr); print('⚠️  Validation FAILED — fix before uploading.')
else:
    print('✅ Validation passed')

## 12. Upload parquet to dev Azure blob

In [ ]:
upload_dev = True
if upload_dev:
    from azure.storage.blob import BlobServiceClient
    connection_string = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
    if not connection_string:
        raise ValueError('Set AZURE_STORAGE_CONNECTION_STRING before uploading.')
    container = os.getenv('AZURE_DEV_CONTAINER', 'parquets-dev')
    blob_name = 'austin-tx-parcels.parquet'
    local_path = os.path.join(DATA_DIR, blob_name)
    if not os.path.exists(local_path):
        raise FileNotFoundError(f'Local parquet not found: {local_path}')
    container_client = BlobServiceClient.from_connection_string(connection_string).get_container_client(container)
    with open(local_path, 'rb') as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)
    print(f'✅ Uploaded {local_path} -> {container}/{blob_name}')
else:
    print('upload_dev is False; skipping upload.')

## 13. Generate PMTiles and upload to dev

H3 low-zoom aggregation (matching Houston). On Windows, tippecanoe runs via WSL2.

In [ ]:
upload_dev_pmtiles = True
if upload_dev_pmtiles:
    import platform, subprocess
    from pathlib import Path
    project_root = Path.cwd()
    while project_root.parent != project_root and not (project_root / 'data' / 'scripts' / 'parquet_to_pmtiles.py').exists():
        project_root = project_root.parent
    script_path = project_root / 'data' / 'scripts' / 'parquet_to_pmtiles.py'
    if not script_path.exists():
        raise FileNotFoundError(f'parquet_to_pmtiles.py not found at {script_path}')
    cmd = [sys.executable, str(script_path), '--city', 'austin', '--h3', '--upload', '--overwrite']
    if not os.getenv('AZURE_STORAGE_CONNECTION_STRING'):
        print('WARNING: AZURE_STORAGE_CONNECTION_STRING not set — running without upload.')
        cmd.remove('--upload')
    if platform.system() == 'Windows':
        cmd.append('--wsl'); print('Windows: adding --wsl (tippecanoe via WSL2)')
    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print('✅ PMTiles done: austin-tx-parcels.pmtiles + -metadata.json')
    else:
        print(f'PMTiles failed (exit {result.returncode})'); print(result.stderr)
else:
    print('upload_dev_pmtiles is False; skipping.')

## 14. Extract parking lots (optional) and upload to dev

`--city austin` is registered in `parking_lot_extraction.py` + `parquet_registry.py`.
NAIP + ML is slow; can also be run from a terminal.

In [ ]:
run_parking = False  # set True to run inline (slow)
if run_parking:
    import subprocess
    from pathlib import Path
    project_root = Path.cwd()
    while project_root.parent != project_root and not (project_root / 'data' / 'scripts' / 'parking_lot_extraction.py').exists():
        project_root = project_root.parent
    script_path = project_root / 'data' / 'scripts' / 'parking_lot_extraction.py'
    cmd = [sys.executable, str(script_path), '--city', 'austin', '--use-osm-supplement']
    if os.getenv('AZURE_STORAGE_CONNECTION_STRING'):
        cmd += ['--upload', '--overwrite']
    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
else:
    print('run_parking is False; run parking_lot_extraction.py --city austin separately.')

## 15. Promote to prod — DO NOT RUN YET

Gated off. Austin stays on **dev only** until Lars reviews
`/app.html?city=austin` and signs off.

In [ ]:
promote_to_prod = False  # leave False until dev sign-off
promote_overwrite = True
PROD_ARTIFACTS = ['austin-tx-parcels.parquet', 'austin-tx-parcels.pmtiles', 'austin-tx-parcels-metadata.json']

if promote_to_prod:
    from azure.storage.blob import BlobServiceClient
    connection_string = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
    if not connection_string:
        raise ValueError('Set AZURE_STORAGE_CONNECTION_STRING before promotion.')
    dev_container  = os.getenv('AZURE_DEV_CONTAINER',  'parquets-dev')
    prod_container = os.getenv('AZURE_PROD_CONTAINER', 'parquets-prod')
    blob_service   = BlobServiceClient.from_connection_string(connection_string)
    for blob_name in PROD_ARTIFACTS:
        dev_blob  = blob_service.get_blob_client(dev_container,  blob_name)
        prod_blob = blob_service.get_blob_client(prod_container, blob_name)
        if not dev_blob.exists():
            print(f'⚠️  Dev blob not found, skipping: {blob_name}'); continue
        if prod_blob.exists():
            if not promote_overwrite:
                raise FileExistsError(f'Prod blob exists: {blob_name}')
            prod_blob.delete_blob()
        prod_blob.start_copy_from_url(dev_blob.url)
        print(f'✅ Promoted {blob_name} -> {prod_container}')
else:
    print('promote_to_prod is False; skipping prod promotion.')